# EY Open Science AI & Data Challenge 2026
## Preprocessing Notebook — EDA, Imputation & Feature Engineering

| | |
|---|---|
| **Notebook purpose** | Exploratory analysis, hierarchical imputation, and feature engineering |
| **Inputs** | `training_with_ee_features_enriched_.csv`, `submission_enriched_.csv` |
| **Outputs** | `train_preprocessed.csv`, `submission_preprocessed.csv` |

---

### Pipeline position

```
EY_Feature_Extraction_Complete.ipynb
        ↓  training_with_ee_features_enriched_.csv
        ↓  submission_enriched_.csv
EY_Preprocessing_Notebook.ipynb          ← you are here
        ↓  train_preprocessed.csv
        ↓  submission_preprocessed.csv
EY_Modeling_Notebook.ipynb
        ↓  final_submission.csv
```

Run all cells **top-to-bottom**. The two output CSVs are consumed by `EY_Modeling_Notebook.ipynb`.


---
## 1. Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import nearest_points
from pyproj import Geod

RANDOM_STATE = 42
print('All imports successful.')


---
## 2. Configuration & Data Loading

The two CSV files below are produced by `EY_Feature_Extraction_Complete.ipynb`.
They already contain all remote-sensing and environmental features assembled across
Pipelines A–D — no joining or horizontal concatenation is needed.

> Set `DATA_DIR` to the folder containing the files if they are not in the current directory.


In [ ]:
DATA_DIR   = '.'  # update if files live elsewhere

water_quality_df = pd.read_csv(os.path.join(DATA_DIR, 'water_quality_training_dataset.csv'))
submission_df = pd.read_csv(os.path.join(DATA_DIR, 'submission_template.csv'))

train_data = pd.read_csv(os.path.join(DATA_DIR, 'training_with_ee_features_enriched_.csv'))
val_data   = pd.read_csv(os.path.join(DATA_DIR, 'submission_enriched_.csv'))

target_columns = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Parse dates
for df in [train_data, val_data]:
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)

# Unique station identifier
for df in [train_data, val_data]:
    df['location_id'] = (
        df['Latitude'].round(4).astype(str) + '_' +
        df['Longitude'].round(4).astype(str)
    )

print(f'Training set:    {train_data.shape[0]:,} rows x {train_data.shape[1]} columns')
print(f'Submission set:  {val_data.shape[0]:,} rows x {val_data.shape[1]} columns')
print(f'Unique train stations:      {train_data["location_id"].nunique()}')
print(f'Unique submission stations: {val_data["location_id"].nunique()}')


**Merging**

In [ ]:
import pandas as pd

def merge_on_geo_date(df1, df2, how='inner'):
    """
    Merge two DataFrames on Longitude, Latitude, and Sample Date.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Must contain 'Longitude', 'Latitude', 'Sample Date'.
    how : str
        Type of merge: 'inner', 'left', 'right', 'outer' (default 'inner').
    Returns
    -------
    pd.DataFrame
        Merged DataFrame.
    """
    # Ensure consistent date parsing
    for df in [df1, df2]:
        df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
        
    merged = pd.merge(
        df1,
        df2,
        on=['Longitude', 'Latitude', 'Sample Date'],
        how=how
    )
    return merged

train_data = merge_on_geo_date(train_data, water_quality_df)
val_data = merge_on_geo_date(val_data, submission_df)


In [ ]:
train_data.isna().sum()

---
## 3. Exploratory Data Analysis

### 3.1 Feature Overview

The dataset spans five environmental data families assembled by the extraction pipeline.
The cell below enumerates all columns and flags the three prediction targets.


In [ ]:
print(f'Total columns: {len(train_data.columns)}\n')
for i, col in enumerate(train_data.columns):
    tag = '  <- TARGET' if col in target_columns else ''
    print(f'  {i+1:3d}. {col}{tag}')


### 3.2 Target Summary Statistics

The three targets differ markedly in scale, skewness, and variance — a core reason
to train **independent models** per target rather than a single multi-output model.


In [ ]:
desc = train_data[target_columns].describe().round(3)
print(desc.to_string())

print('\nSkewness:')
for t in target_columns:
    print(f'  {t:42s}: {train_data[t].skew():+.3f}')


### 3.3 Geographic Distribution

Blue = training stations. Red = submission stations.
The geographic gap between the two sets is the dominant modelling challenge:
submission stations are clustered along the southern coast while training stations
span the whole country.


In [ ]:
center = [train_data['Latitude'].mean(), train_data['Longitude'].mean()]
m = folium.Map(location=center, zoom_start=5, tiles='CartoDB positron')

for _, row in train_data.drop_duplicates('location_id').iterrows():
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=4, color='steelblue', fill=True, fill_opacity=0.7,
        tooltip='Train'
    ).add_to(m)

for _, row in val_data.drop_duplicates('location_id').iterrows():
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=5, color='crimson', fill=True, fill_opacity=0.8,
        tooltip='Submission'
    ).add_to(m)

m


### 3.4 Target Distributions

Histograms reveal the right skew of Dissolved Reactive Phosphorus, the wide
range of Electrical Conductance, and the roughly bimodal structure of Total
Alkalinity. Log-scale boxplots highlight outlier structure and spread.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col_idx, target in enumerate(target_columns):
    data = train_data[target].dropna()

    # histogram
    axes[0, col_idx].hist(data, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[0, col_idx].set_title(target, fontweight='bold')
    axes[0, col_idx].set_xlabel('Value'); axes[0, col_idx].set_ylabel('Count')

    # log-scale boxplot
    axes[1, col_idx].boxplot(
        data, patch_artist=True,
        boxprops=dict(facecolor='steelblue', alpha=0.7),
        medianprops=dict(color='white', linewidth=2),
        flierprops=dict(marker='.', markersize=3, alpha=0.3)
    )
    axes[1, col_idx].set_yscale('log')
    axes[1, col_idx].set_title(f'{target} (log scale)')
    axes[1, col_idx].set_ylabel('Value (log)')

plt.suptitle('Target Distributions — Training Set', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


### 3.5 Pairwise Target Correlations

If the targets were highly correlated a single multi-output model would be a natural
choice. Scatter plots and Pearson r show the actual co-dependence structure.


In [ ]:
pairs = [
    ('Total Alkalinity',       'Electrical Conductance'),
    ('Total Alkalinity',       'Dissolved Reactive Phosphorus'),
    ('Electrical Conductance', 'Dissolved Reactive Phosphorus'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (x, y) in zip(axes, pairs):
    sub = train_data[[x, y]].dropna()
    r   = sub.corr().iloc[0, 1]
    ax.scatter(sub[x], sub[y], alpha=0.2, s=10, color='steelblue')
    ax.set_xlabel(x, fontsize=8); ax.set_ylabel(y, fontsize=8)
    ax.set_title(f'Pearson r = {r:.3f}', fontsize=11, fontweight='bold')

plt.suptitle('Pairwise Target Scatter Plots', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Target correlation matrix:')
print(train_data[target_columns].corr().round(3).to_string())


### 3.6 Missing Value Analysis

Remote-sensing features (Landsat, ERA5-Land) frequently show gaps due to cloud
cover or sensor scheduling windows. The bar chart ranks features by missing rate;
the heatmap reveals whether missingness is scattered or clustered in specific
observation windows.


In [ ]:
missing_pct = train_data.isna().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Bar chart
missing_pct.plot.bar(ax=axes[0], color='coral', edgecolor='white', linewidth=0.3)
axes[0].axhline(0.10, color='crimson', linestyle='--', linewidth=1.2, label='10% threshold')
axes[0].set_title('Missing Value Rate by Feature', fontweight='bold')
axes[0].set_ylabel('Fraction missing')
axes[0].tick_params(axis='x', rotation=90)
axes[0].legend()

# Missingness heatmap
high_miss = missing_pct[missing_pct > 0.05].index.tolist()
if high_miss:
    sample = train_data[high_miss].head(300).isna()
    sns.heatmap(sample.T, cbar=False, ax=axes[1],
                cmap='Reds', yticklabels=True, xticklabels=False)
    axes[1].set_title('Missingness Pattern (first 300 obs, features > 5% missing)',
                      fontweight='bold')
    axes[1].set_xlabel('Observations ->')
else:
    axes[1].text(0.5, 0.5, 'No feature exceeds 5% missing',
                ha='center', transform=axes[1].transAxes)

plt.tight_layout(); plt.show()

print(f'Features with any missing values:  {len(missing_pct)}')
print(f'Features with > 10% missing:       {(missing_pct > 0.10).sum()}')


### 3.7 Temporal Trends

Monthly averages plotted over the full observation window, with a 3-month rolling
mean (red dashed) to highlight low-frequency trends.


In [ ]:
td = train_data.copy()
td['Year']  = td['Sample Date'].dt.year
td['Month'] = td['Sample Date'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, target in zip(axes, target_columns):
    monthly = td.groupby(['Year','Month'])[target].mean().reset_index()
    monthly['date'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))
    monthly = monthly.sort_values('date')
    ax.plot(monthly['date'], monthly[target], color='steelblue', linewidth=1.2, alpha=0.8)
    ax.fill_between(monthly['date'], monthly[target], alpha=0.12, color='steelblue')
    ax.plot(monthly['date'], monthly[target].rolling(3, center=True).mean(),
            color='crimson', linewidth=2, linestyle='--', label='3-month avg')
    ax.set_title(target, fontsize=10, fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('Mean value')
    ax.legend(fontsize=8); ax.tick_params(axis='x', rotation=30)

plt.suptitle('Monthly Average Target Values Over Time',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### 3.8 Seasonal Patterns — Monthly Distribution Spread

Monthly boxplots show the **full distribution** within each calendar month, not
just the mean. Wide boxes in austral summer (Nov–Feb) for DRP, for example, reflect
high event-driven variability from storm runoff during the rainy season.


In [ ]:
td['Month'] = td['Sample Date'].dt.month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, target in zip(axes, target_columns):
    groups = [td[td['Month'] == m][target].dropna().values for m in range(1, 13)]
    bp = ax.boxplot(groups, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='.', markersize=3, alpha=0.3))
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels, rotation=45, fontsize=8)
    ax.set_title(target, fontsize=10, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Seasonal Distribution of Targets by Month (All Stations)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### 3.9 Station-Level Variability

Some monitoring stations show unusually large temporal swings — driven by proximity
to agricultural runoff, intermittent rivers, or storm-prone catchments. The
coefficient of variation (CV = σ/μ) identifies the most volatile stations.
Understanding which stations are volatile helps calibrate trust in imputed values.


In [ ]:
station_stats = (
    train_data.groupby('location_id')[target_columns]
    .agg(['mean', 'std', 'count'])
)
station_stats.columns = ['_'.join(c) for c in station_stats.columns]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, target in zip(axes, target_columns):
    mean_c = f'{target}_mean'; std_c = f'{target}_std'
    cv     = station_stats[std_c] / (station_stats[mean_c].abs() + 1e-6)
    top15  = cv.sort_values(ascending=False).head(15)
    ax.barh(range(len(top15)), top15.values, color='steelblue', edgecolor='white')
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels([s[-16:] for s in top15.index], fontsize=7)
    ax.set_title(f'Top 15 most variable stations\n{target[:26]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Coefficient of Variation (sigma/mu)')

plt.suptitle('Station-Level Temporal Variability', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### 3.10 Spatial Heatmap — Mean Target Values per Station

Geographic structure is prominent: Total Alkalinity and Electrical Conductance peak
in the dry interior (Karoo, Highveld) and decrease towards the wetter coasts.
Black crosses mark submission station locations.


In [ ]:
station_means = (
    train_data.groupby('location_id')[target_columns + ['Latitude', 'Longitude']]
    .mean().reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, target in zip(axes, target_columns):
    vals = station_means[target]
    norm = mcolors.Normalize(vmin=vals.quantile(0.05), vmax=vals.quantile(0.95))
    cmap = cm.RdYlGn_r if target == 'Dissolved Reactive Phosphorus' else cm.RdYlBu_r
    sc = ax.scatter(station_means['Longitude'], station_means['Latitude'],
                    c=vals, cmap=cmap, norm=norm, s=80,
                    edgecolors='white', linewidths=0.4, alpha=0.9)
    ax.scatter(val_data['Longitude'], val_data['Latitude'],
               marker='x', color='black', s=40, linewidths=1.2,
               label='Submission stations', alpha=0.7)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.04)
    ax.set_title(f'{target}\n(mean per station)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.legend(fontsize=8)

plt.suptitle('Spatial Distribution of Mean Target Values — Training Stations',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


---
## 4. Distance-to-Sea Feature

South Africa's water chemistry is influenced by proximity to the coast — marine
aerosols, wetter microclimates, and fog-driven weathering all attenuate inland.
We compute geodesic distance (km) to the nearest Natural Earth 10 m coastline
segment, then derive two transforms:

| Feature | Formula | Purpose |
|---|---|---|
| `distance_to_sea_km` | Geodesic distance | Raw geographic feature |
| `log_distance_sea` | log(1 + d) | Stabilises large inland values |
| `marine_influence` | 1/(d + 1) | Soft proximity weight (overwritten in feature engineering) |


In [ ]:
geod       = Geod(ellps='WGS84')
coastline  = gpd.read_file('ne_10m_coastline/ne_10m_coastline.shp').to_crs('EPSG:4326')
coast_union = coastline.geometry.union_all()


def distance_to_sea(lat, lon):
    point        = Point(lon, lat)
    nearest_geom = nearest_points(point, coast_union)[1]
    _, _, dist_m = geod.inv(lon, lat, nearest_geom.x, nearest_geom.y)
    return dist_m / 1000


def add_distance_to_sea(df):
    df = df.copy()
    df['distance_to_sea_km'] = df.apply(
        lambda r: distance_to_sea(r['Latitude'], r['Longitude']), axis=1
    )
    df['log_distance_sea'] = np.log1p(df['distance_to_sea_km'])
    df['marine_influence']  = 1 / (df['distance_to_sea_km'] + 1)
    return df


print('Computing distances for training set (may take ~1 min)...')
train_data = add_distance_to_sea(train_data)
print('Computing distances for submission set...')
val_data   = add_distance_to_sea(val_data)

print('Added: distance_to_sea_km, log_distance_sea, marine_influence')
print(f'  Train range: {train_data["distance_to_sea_km"].min():.1f} - '
      f'{train_data["distance_to_sea_km"].max():.1f} km')
print(f'  Sub   range: {val_data["distance_to_sea_km"].min():.1f} - '
      f'{val_data["distance_to_sea_km"].max():.1f} km')


---
## 5. Feature–Target Correlation Analysis

With all base features (including distance-to-sea) now present, we rank features
by absolute correlation with each target. This analysis motivates the curated
per-target feature sets used in the modelling notebook.

> **Note on spatial overfitting:** `distance_to_sea_km` correlates moderately
> with TA (r ≈ +0.53) and EC (r ≈ +0.44). Using the raw distance as a model
> feature caused the model to memorise training station geography rather than
> generalise to the southern coast submission zone. We therefore replace it with
> an exponential decay transform `exp(−d / 100)` during feature engineering.


In [ ]:
df_num   = train_data.select_dtypes(include='number')
corr_mat = df_num.corr()[target_columns].copy()
corr_mat['combined_abs'] = corr_mat.abs().sum(axis=1)
corr_sorted = corr_mat.sort_values('combined_abs', ascending=False)

print('Top 20 features by combined absolute correlation with all three targets:')
print(corr_sorted.head(20).round(3).to_string())

# Visual: top-15 per target
feat_pool = df_num.drop(columns=target_columns, errors='ignore')
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, target in zip(axes, target_columns):
    corr = feat_pool.corrwith(train_data[target]).abs().sort_values(ascending=False).head(15)
    corr.sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Top 15 features\n|r| with {target[:22]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('|Pearson r|')

plt.suptitle('Feature-Target Correlations (including distance-to-sea)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


---
## 6. Hierarchical Spatio-Temporal Imputation

### Rationale

Remote-sensing features — particularly Landsat bands (cloud gaps) and
ERA5-Land daily fields — can be missing for specific station-dates.
Simple global-mean imputation would overwrite the strong spatial and temporal
structure in these gaps with a single uninformative value.

We use a **5-level cascade**: missing values are first filled from fine-grained
group means, falling back to coarser aggregations only when no finer-level data
is available.

| Level | Grouping | Captures |
|---|---|---|
| 1 | Year × Month × Station | Within-season station behaviour |
| 2 | Month × Station | Long-term seasonal station profile |
| 3 | Station | Long-term station mean |
| 4 | Month | National seasonal signal |
| 5 | Global mean | Last-resort fallback |

Group means are **fitted on the training set only** and applied to both
train and submission sets — this ensures no label leakage.


In [ ]:
def fit_hierarchical_means(train_df):    
    df = train_df.copy()
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)

    df['Year'] = df['Sample Date'].dt.year
    df['Month'] = df['Sample Date'].dt.month

    numeric_cols = df.select_dtypes(include='number').columns

    means = {
        "YMLL": df.groupby(['Year','Month','Latitude','Longitude'])[numeric_cols].mean(),
        "MLL":  df.groupby(['Month','Latitude','Longitude'])[numeric_cols].mean(),
        "LL":   df.groupby(['Latitude','Longitude'])[numeric_cols].mean(),
        "M":    df.groupby(['Month'])[numeric_cols].mean(),
        "global": df[numeric_cols].mean()
    }

    return means


def hierarchical_impute(df, means):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)

    df['Year'] = df['Sample Date'].dt.year
    df['Month'] = df['Sample Date'].dt.month

    numeric_cols = df.select_dtypes(include='number').columns

    # --- 1. Merge Y-M-Lat-Lon ---
    df = df.merge(
        means["YMLL"],
        how="left",
        left_on=['Year','Month','Latitude','Longitude'],
        right_index=True,
        suffixes=("", "_YMLL")
    )

    # --- 2. Merge M-Lat-Lon ---
    df = df.merge(
        means["MLL"],
        how="left",
        left_on=['Month','Latitude','Longitude'],
        right_index=True,
        suffixes=("", "_MLL")
    )

    # --- 3. Merge Lat-Lon ---
    df = df.merge(
        means["LL"],
        how="left",
        left_on=['Latitude','Longitude'],
        right_index=True,
        suffixes=("", "_LL")
    )

    # --- 4. Merge Month ---
    df = df.merge(
        means["M"],
        how="left",
        left_on=['Month'],
        right_index=True,
        suffixes=("", "_M")
    )

    # --- 5. Global fallback ---
    global_vals = means["global"]

    # --- Remplissage hiérarchique vectorisé ---
    for col in numeric_cols:
        df[col] = (
            df[col]
            .fillna(df[col + "_YMLL"])
            .fillna(df[col + "_MLL"])
            .fillna(df[col + "_LL"])
            .fillna(df[col + "_M"])
            .fillna(global_vals[col])
        )

    # Nettoyage des colonnes temporaires
    drop_cols = [c for c in df.columns if any(c.endswith(s) for s in ["_YMLL","_MLL","_LL","_M"])]
    df.drop(columns=drop_cols, inplace=True)

    return df


In [ ]:
# Strategy note: we retain all stations and use hierarchical imputation
# rather than dropping stations with high missingness. Leaderboard evaluation
# confirmed this consistently outperformed any station-dropping approach.

means_full = fit_hierarchical_means(train_data)

train_imputed      = hierarchical_impute(train_data, means_full)
submission_imputed = hierarchical_impute(val_data,   means_full)

print('Imputation complete.')
print(f'  Missing (Train): {train_imputed.isna().sum().sum()} | Shape: {train_imputed.shape}')
print(f'  Missing (Sub):   {submission_imputed.isna().sum().sum()} | Shape: {submission_imputed.shape}')


---
## 7. Feature Engineering

Feature engineering was the **highest-impact step** in the pipeline. Raw environmental
measurements were transformed into a rich feature matrix capturing multi-scale
precipitation dynamics, terrain-climate interactions, marine influence, and cyclical
temporal encoding.

| Family | Examples | Physical motivation |
|---|---|---|
| Temporal (cyclical) | `Month_sin/cos`, `DayOfYear_cos` | Preserves circular continuity of seasonal signal |
| Spectral interactions | `NDMI x MNDWI`, `NDMI^2` | Non-linear moisture index combinations |
| PET transforms | `pet^2`, `log(pet)`, `pet x Lat` | PET is right-skewed; interactions encode climate zones |
| Precipitation windows | `precip_rollsum_2..14` | Antecedent moisture at multiple lag scales |
| Terrain-climate | `TWI x precip`, `runoff_index` | Topographic amplification of runoff |
| Marine influence | `exp(-d/100)`, `coastal_zone` | Smooth spatial proxy replacing raw distance |
| Soil interactions | `soil_cec x soil_carbon` | Nutrient cycling and retention capacity |
| Land use x climate | `ag_pct x precip`, `urban_pct x precip_30d` | Diffuse pollution loading potential |


In [ ]:
def advanced_feature_engineering(df):
    """
    Feature engineering step.

    NOTE — REDACTED FOR THIS PUBLIC REPOSITORY.
    The exact engineered features, interaction terms, rolling-window
    configurations, and regime-binning thresholds used in our competition
    entry have been withheld, since this pipeline is part of ongoing
    (unpublished) research we intend to continue.

    At a high level, this step:
      - derives calendar/cyclical time features from `Sample Date`
      - builds interaction and polynomial terms from the remote-sensing
        spectral indices (NDMI, MNDWI) and climate variables (PET, precipitation)
      - discretizes select climate/terrain variables into quantile-based
        regime bins (e.g. wetness, aridity, terrain wetness index)
      - computes rolling precipitation statistics per station over multiple
        lag windows to capture antecedent moisture conditions
      - derives a marine-influence proxy from distance-to-coastline
      - builds terrain x climate and land-cover x climate interaction terms

    See docs/Model_Description.md for the general methodology description.
    """
    df = df.copy()
    raise NotImplementedError(
        "Feature engineering implementation redacted in the public repository. "
        "See docs/Model_Description.md for the high-level methodology."
    )


In [ ]:
train_data.head()

In [ ]:
train_feat      = advanced_feature_engineering(train_imputed)
submission_feat = advanced_feature_engineering(submission_imputed)

feature_columns = [
    c for c in train_feat.columns
    if c not in target_columns + ['Sample Date']
]

print(f'Columns after engineering: {train_feat.shape[1]}')
print(f'Feature columns:           {len(feature_columns)}')
print(f'Training rows:    {train_feat.shape[0]:,}')
print(f'Submission rows:  {submission_feat.shape[0]:,}')


---
## 8. Save Preprocessed Data

These two CSV files are the inputs consumed by `EY_Modeling_Notebook.ipynb`.
The training file contains all engineered features **and** the target columns.
The submission file contains only features (no targets — they are what we predict).


In [ ]:
TRAIN_PREP = 'train_preprocessed.csv'
SUB_PREP   = 'submission_preprocessed.csv'

train_feat.to_csv(TRAIN_PREP,  index=False)
submission_feat.to_csv(SUB_PREP, index=False)

print(f'Saved: {TRAIN_PREP}     {train_feat.shape}')
print(f'Saved: {SUB_PREP}  {submission_feat.shape}')
print()
print('Next step: open EY_Modeling_Notebook.ipynb')
